In [ ]:
import sys
sys.path.insert(0, '..')

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats

from src.config import DB_PATH

plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query("""
        SELECT username, date, tournament_slug, rank, score, aroc_1, performance_rating
        FROM titled_tuesday_standings
        WHERE performance_rating IS NOT NULL
          AND score > 0
          AND wins + draws + byes >= 5      -- exclude early dropouts
    """, conn)

df['date'] = pd.to_datetime(df['date'])
print(f"{len(df):,} rows  |  {df['tournament_slug'].nunique()} tournaments  |  {df['username'].nunique()} players")
df.describe().round(1)

## Overall Correlation

In [ ]:
r,   pval_r   = stats.pearsonr(df['score'], df['performance_rating'])
rho, pval_rho = stats.spearmanr(df['score'], df['performance_rating'])
print(f"Pearson r  = {r:.4f}   p = {pval_r:.2e}")
print(f"Spearman ρ = {rho:.4f}   p = {pval_rho:.2e}")

# The formula guarantees a linear relationship; what's interesting is the scatter
# around it — that's the field-strength signal.
slope, intercept, *_ = stats.linregress(df['score'], df['performance_rating'])
print(f"\nOLS: performance_rating = {intercept:.1f} + {slope:.1f} × score")
print(f"Implied Elo per point scored: {slope:.1f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── left: hexbin density ─────────────────────────────────────────────────────
ax = axes[0]
hb = ax.hexbin(df['score'], df['performance_rating'], gridsize=50,
               cmap='YlOrRd', mincnt=1, linewidths=0.2)
fig.colorbar(hb, ax=ax, label='Player-tournament count')

xs = np.linspace(df['score'].min(), df['score'].max(), 200)
ax.plot(xs, intercept + slope * xs, 'b--', lw=1.5, label=f'OLS (r={r:.3f})')
ax.set_xlabel('Score (out of 11)')
ax.set_ylabel('Performance Rating')
ax.set_title('Score vs Performance Rating (all players)')
ax.legend(fontsize=9)

# ── right: marginal distributions ────────────────────────────────────────────
ax2 = axes[1]
ax3 = ax2.twinx()
ax2.hist(df['score'], bins=np.arange(0, 11.6, 0.5), color='steelblue',
         alpha=0.6, label='Score')
ax3.hist(df['performance_rating'], bins=60, color='tomato',
         alpha=0.5, label='Performance Rating')
ax2.set_xlabel('Value')
ax2.set_ylabel('Count (score)', color='steelblue')
ax3.set_ylabel('Count (perf rating)', color='tomato')
ax2.set_title('Marginal Distributions')
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax3.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, fontsize=9)

plt.tight_layout()
plt.show()

## Same Score, Different Field: Performance Rating Spread by Score Level

The core question: for a fixed score, how much does performance rating vary?

In [ ]:
score_levels = [5.0, 5.5, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0, 10.5, 11.0]
df_top = df[df['score'].isin(score_levels)].copy()

# Summary: std of performance_rating at each score level
spread = (df_top.groupby('score')['performance_rating']
          .agg(['count','mean','std','min','max'])
          .rename(columns={'count':'n','mean':'mean_pr','std':'std_pr','min':'min_pr','max':'max_pr'})
          .round(1))
spread['range_pr'] = spread['max_pr'] - spread['min_pr']
print(spread.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

sns.violinplot(
    data=df_top, x='score', y='performance_rating',
    order=score_levels, palette='coolwarm',
    inner='box', cut=0, ax=ax,
)

# Overlay the formula's expected value (aroc_1 cancels; this is just a
# reference line showing that the mean at each score equals roughly the
# mean aroc + 800*(score/11 - 0.5) — i.e., the mean PR tracks the mean AROC.)
means = df_top.groupby('score')['performance_rating'].mean()
ax.plot(range(len(score_levels)), [means.get(s, np.nan) for s in score_levels],
        'ko--', ms=4, lw=1, label='Mean PR')

ax.set_xlabel('Score (out of 11)')
ax.set_ylabel('Performance Rating')
ax.set_title('Performance Rating Distribution for Each Score Level\n'
             'Width shows how much field strength varies for the same score')
ax.legend()
plt.tight_layout()
plt.show()

## Field Strength Over Time

How does average AROC 1 (field strength) vary tournament-to-tournament?

In [ ]:
field = (
    df.groupby(['date', 'tournament_slug'])
    .agg(
        median_aroc=('aroc_1', 'median'),
        mean_aroc=('aroc_1', 'mean'),
        top10_aroc=('aroc_1', lambda x: x.nlargest(10).mean()),
        n_players=('username', 'count'),
    )
    .reset_index()
    .sort_values('date')
)

fig, ax = plt.subplots(figsize=(15, 5))

ax.plot(field['date'], field['median_aroc'], color='steelblue', lw=1.2,
        alpha=0.7, label='Median AROC 1')
ax.plot(field['date'], field['top10_aroc'], color='tomato', lw=1.2,
        alpha=0.7, label='Top-10 avg AROC 1')

# 4-tournament rolling mean
ax.plot(field['date'], field['median_aroc'].rolling(4, min_periods=2).mean(),
        color='navy', lw=2, label='Median 4-tourn rolling avg')

ax.set_xlabel('Tournament Date')
ax.set_ylabel('AROC 1 (avg opponent Elo)')
ax.set_title('Field Strength Over Time (AROC 1)')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\nWeakest field:  {field.loc[field['median_aroc'].idxmin(), 'tournament_slug']}  "
      f"({field['median_aroc'].min():.0f})")
print(f"Strongest field: {field.loc[field['median_aroc'].idxmax(), 'tournament_slug']}  "
      f"({field['median_aroc'].max():.0f})")

## Score vs Performance Rating Residuals

After removing the linear score effect, the residual captures **pure field strength**: how much harder or easier was this performance than average for the same score?

In [ ]:
# Residual = performance_rating - (intercept + slope * score)
df['pr_residual'] = df['performance_rating'] - (intercept + slope * df['score'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── left: residual vs aroc_1 ─────────────────────────────────────────────────
ax = axes[0]
hb = ax.hexbin(df['aroc_1'], df['pr_residual'], gridsize=50,
               cmap='YlOrRd', mincnt=1, linewidths=0.2)
fig.colorbar(hb, ax=ax, label='Count')
ax.axhline(0, color='black', lw=1, ls='--')
r2, _ = stats.pearsonr(df['aroc_1'], df['pr_residual'])
ax.set_xlabel('AROC 1 (avg opponent Elo)')
ax.set_ylabel('PR Residual (field-strength component)')
ax.set_title(f'AROC 1 vs Residual  (r={r2:.3f})')

# ── right: distribution of residuals ────────────────────────────────────────
ax2 = axes[1]
ax2.hist(df['pr_residual'], bins=80, color='steelblue', alpha=0.7, edgecolor='none')
ax2.axvline(0, color='black', lw=1.5, ls='--')
ax2.set_xlabel('PR Residual')
ax2.set_ylabel('Count')
ax2.set_title(f'Residual Distribution\nstd = {df["pr_residual"].std():.1f} Elo pts')

plt.tight_layout()
plt.show()

print(f"Residual std: {df['pr_residual'].std():.1f} Elo points")
print(f"This means same score can differ by ~{df['pr_residual'].std()*2:.0f} Elo pts (±1 std) due to field strength alone")

## Player-Level: Score vs Performance Rating

For regular TT attendees, compare each player's **average score** vs **average performance rating**. Players above the line punch above their field strength; below the line they tend to play in weaker fields.

In [ ]:
MIN_APPEARANCES = 15

player_agg = (
    df.groupby('username')
    .agg(
        n=('score', 'count'),
        mean_score=('score', 'mean'),
        mean_pr=('performance_rating', 'mean'),
        mean_aroc=('aroc_1', 'mean'),
    )
    .query('n >= @MIN_APPEARANCES')
    .reset_index()
)

# OLS on player averages
sl2, int2, *_ = stats.linregress(player_agg['mean_score'], player_agg['mean_pr'])
player_agg['pr_vs_expected'] = player_agg['mean_pr'] - (int2 + sl2 * player_agg['mean_score'])

fig, ax = plt.subplots(figsize=(10, 8))

norm = mcolors.Normalize(vmin=player_agg['mean_aroc'].quantile(0.05),
                          vmax=player_agg['mean_aroc'].quantile(0.95))
sc = ax.scatter(player_agg['mean_score'], player_agg['mean_pr'],
                c=player_agg['mean_aroc'], cmap='plasma', norm=norm,
                s=player_agg['n'] / 2, alpha=0.7, edgecolors='none')
plt.colorbar(sc, ax=ax, label='Mean AROC 1')

xs2 = np.linspace(player_agg['mean_score'].min(), player_agg['mean_score'].max(), 200)
ax.plot(xs2, int2 + sl2 * xs2, 'k--', lw=1.5, label='OLS')

# Label notable outliers (most above / below the line)
top_over  = player_agg.nlargest(8, 'pr_vs_expected')
top_under = player_agg.nsmallest(8, 'pr_vs_expected')
for _, row in pd.concat([top_over, top_under]).iterrows():
    ax.annotate(row['username'], (row['mean_score'], row['mean_pr']),
                fontsize=7, alpha=0.8,
                xytext=(4, 2), textcoords='offset points')

ax.set_xlabel('Mean Score per Tournament')
ax.set_ylabel('Mean Performance Rating')
ax.set_title(f'Player Averages: Score vs Performance Rating\n'
             f'(bubble size ∝ appearances; colour = mean AROC 1; n≥{MIN_APPEARANCES})')
ax.legend()
plt.tight_layout()
plt.show()

print("\nPlayers who most outperform their score (play in tougher fields):")
print(top_over[['username','n','mean_score','mean_pr','mean_aroc','pr_vs_expected']].to_string(index=False))
print("\nPlayers who most underperform their score (benefit from weaker fields):")
print(top_under[['username','n','mean_score','mean_pr','mean_aroc','pr_vs_expected']].to_string(index=False))